# Load Address Prediction Results

This notebook reads `results.csv` and plots runtime vs. iterations for each access-pattern and core-class combination.

In [1]:
import altair as alt
import duckdb

In [4]:
con = duckdb.connect()

df = con.sql("""
    SELECT
        iters AS iterations,
        pattern,
        core_kind,
        median,
        unit,
        cpu
    FROM 'results.csv'
    ORDER BY core_kind, pattern, iterations
""").df()

df.head()

,iterations,pattern,core_kind,median,unit,cpu
0,10,random,ecore,4502,cycles,0
1,20,random,ecore,4565,cycles,0
2,30,random,ecore,4666,cycles,0
3,40,random,ecore,4747,cycles,1
4,50,random,ecore,4839,cycles,1


In [5]:
sa_rv_df = df[df["pattern"].isin(["strided", "sa_rv", "random"])]
unit = df["unit"].iloc[0]
y_title = f"Runtime ({unit})"
sa_rv_chart = (
    alt.Chart(sa_rv_df)
    .mark_line(point=True)
    .encode(
        x=alt.X("iterations:Q", title="Iterations"),
        y=alt.Y("median:Q", title=y_title),
        color=alt.Color("pattern:N", title="Pattern"),
        tooltip=[
            "pattern", "core_kind", "iterations", "median","cpu"
            # "min", "p25", "p75", "max",
            # "first", "last", "unit",
        ],
    )
    .properties(width=380, height=280)
    .facet(column=alt.Column("core_kind:N", title="Core Class", sort=["ecore", "pcore"]))
    .properties(title="Strided vs. SA+RV — if they match, speedup is address (not value) prediction")
    .interactive()
)

sa_rv_chart

alt.FacetChart(...)